In [1]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
# from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os
from euriai.langchain import create_chat_model
import time

app_dir = os.path.join(os.getcwd(), "app")
load_dotenv(os.path.join(app_dir, ".env"))

api_key = os.getenv("key")

chat_model = create_chat_model(api_key=api_key, model="gpt-4.1-nano", temperature=0.7)

c:\EGA\code\RAG\Udemy_RAG_1\Udemy-Advanced-LangChain\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
c:\EGA\code\RAG\Udemy_RAG_1\Udemy-Advanced-LangChain\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
a = PromptTemplate(template="Write a joke about the following - {item}", input_variables=["item"],)
a.invoke({"item":"Icecream"})
model = chat_model
parser = StrOutputParser()

chain = a | model | parser

chain.invoke({"item":"Icecream"})

'Why did the ice cream go to school? Because it wanted to be a little "sundae" smarter!'

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnablePassthrough, RunnableSequence

chain = RunnablePassthrough()
chain.invoke({"name":"abhijeet"})

{'name': 'abhijeet'}

In [4]:
chain = RunnableParallel(
    {
        "output" : RunnableLambda(lambda x : x.upper()),
        "input" : RunnablePassthrough()
    }
)

chain.invoke("Abhijeet")

{'output': 'ABHIJEET', 'input': 'Abhijeet'}

In [28]:
chain = RunnableParallel(
    {
        "x": RunnablePassthrough()
    }
).assign(y = RunnableLambda(lambda x : x["x"].upper()))

chain.invoke("hi")

{'x': 'hi', 'y': 'HI'}

In [29]:
c = RunnablePassthrough().assign(y=lambda x:x["x"].upper())
c.invoke({"x":"Anshu"})

{'x': 'Anshu', 'y': 'ANSHU'}

In [ ]:
def assign_f():
    return 100

chain = RunnableParallel(
    {
        "x":RunnableLambda(lambda x: 100),
        "y":RunnableLambda(lambda y: 200)
    }
)

chain.invoke()

{'x': 100, 'y': 200}

In [5]:
from langchain_core.documents.base import Document
from langchain_chroma import Chroma
from langchain_core.runnables import RunnablePassthrough

In [6]:
## RAG LECL

from euriai.langchain import EuriaiEmbeddings

docs1 = Document(page_content="the dog loves to eat pizza", metadata={"source":"animal.txt", "type":"domestic"})
docs2 = Document(page_content="the cat loves to eat milk", metadata={"source":"animal.txt", "type":"domestic"})

final_docs = [docs1, docs2]

embedding_function = EuriaiEmbeddings(api_key=api_key, model="text-embedding-3-small")

In [7]:
db = Chroma.from_documents(final_docs, embedding_function)

In [8]:
retriver = db.as_retriever()

In [9]:
retriver.invoke("What does the dog want to eat?")

[Document(id='1dbc7c76-03d2-4020-90a3-210b74d2ce3b', metadata={'type': 'domestic', 'source': 'animal.txt'}, page_content='the dog loves to eat pizza'),
 Document(id='6dfc6f32-87fb-4ea0-893d-22083f389ec7', metadata={'source': 'animal.txt', 'type': 'domestic'}, page_content='the cat loves to eat milk')]

In [10]:
template = """Answer the following question based on the following context
{context}

Question: {question}"""

prompt = PromptTemplate(template= template, input_variables=["context", "question"])
prompt

PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the following question based on the following context\n{context}\n\nQuestion: {question}')

In [22]:
c1 = RunnableParallel(
    {
        "question" : RunnableLambda(lambda x: x["question"]),
        "context" : RunnableLambda(lambda x: x["question"]) | retriver
    }
) | prompt | chat_model | StrOutputParser()

c1.invoke({"question":"What does the dog eat ?"})

'The dog loves to eat pizza.'

In [21]:
c1 = (
    RunnableParallel({"question": RunnablePassthrough(),"context": retriver})
    | prompt
    | chat_model
    | StrOutputParser()
)

c1.invoke("What does the dog like to eat?")

'The dog likes to eat pizza.'

In [19]:

from operator import itemgetter

c1 = (
    RunnableParallel(
        {
            "question": itemgetter("question"),
            "context": itemgetter("question") | retriver
        }
    )
    | prompt
    | chat_model
    | StrOutputParser()
)


c1.invoke({"question":"What does the cat eat ?"})

'The cat loves to eat milk.'